# 🎯 Ultimate Classroom Emotion Recognition — AffectNet + FER2013 + DAiSEE

**Pipeline tốt nhất cho Emotion Recognition trong lớp học.**

### 3-Dataset Pretrain Strategy
1. **AffectNet** (~420K images, 8 classes) → Học facial features mạnh
2. **FER2013** (~35K images) → Bổ sung thêm dữ liệu
3. **DAiSEE** (video clips) → Fine-tune cho classroom context

### Cải tiến chính
| # | Cải tiến | Chi tiết |
|---|---|---|
| 1 | AffectNet pretrain | ~60K samples từ dataset lớn nhất thế giới |
| 2 | Adaptive extraction | 40-50 frames/clip cho class thiểu số |
| 3 | Offline augmentation | Mỗi class ≥ 2500 samples |
| 4 | 3-stage fine-tune | Head → 30% → 60% backbone |
| 5 | Cosine decay LR | Hội tụ mượt hơn |
| 6 | Focal Loss γ=2.5 | Tập trung hard examples |
| 7 | Stronger head | 512→256→4 với BN+Dropout mạnh |

> **Runtime:** Chọn **GPU** (T4/A100/V100) trong Runtime → Change runtime type


In [ ]:
# ============================================================
# CELL 1 — Environment Setup
# ============================================================
import os, subprocess

res = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if res.returncode == 0:
    for line in res.stdout.strip().splitlines()[:8]:
        print(line)
else:
    print('\u26a0\ufe0f GPU NOT FOUND! Runtime > Change runtime type > GPU')

!pip install -q kaggle opencv-python-headless scikit-learn seaborn tqdm albumentations
print('\n\u2705 Environment ready.')

In [ ]:
# ============================================================
# CELL 2 — Configuration (CHỈNH S\u1eeca \u1ede \u0110\u00c2Y)
# ============================================================
import random
import numpy as np
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ========== TOGGLES ==========
USE_AFFECTNET   = True    # Pretrain tr\u00ean AffectNet (recommended)
USE_FER_PRETRAIN = True   # Pretrain th\u00eam tr\u00ean FER2013

# ========== AffectNet ==========
AFFECTNET_KAGGLE = 'manavukani/affectnet'      # Dataset Kaggle
AFFECTNET_MAX_PER_CLASS = 15000  # Max images/class t\u1eeb AffectNet
AFFECTNET_EPOCHS_HEAD     = 8
AFFECTNET_EPOCHS_FINETUNE = 10

# ========== FER2013 ==========
FER_EPOCHS_HEAD     = 6
FER_EPOCHS_FINETUNE = 6

# ========== DAiSEE ==========
DAISEE_EPOCHS_HEAD = 15
DAISEE_EPOCHS_FT1  = 20   # fine-tune 30% backbone
DAISEE_EPOCHS_FT2  = 15   # fine-tune 60% backbone

# ========== GENERAL ==========
IMG_SIZE   = 260
BATCH_SIZE = 32
NUM_CLASSES = 4

# ========== ADAPTIVE FRAME EXTRACTION ==========
FRAMES_PER_CLIP = {
    0: 40,   # Bu\u1ed3n ch\u00e1n  (r\u1ea5t \u00edt clips)
    1: 5,    # T\u1eadp trung (r\u1ea5t nhi\u1ec1u clips)
    2: 12,   # H\u1ee9ng th\u00fa  (trung b\u00ecnh)
    3: 50,   # B\u00ecnh th\u01b0\u1eddng  (c\u1ef1c \u00edt clips)
}
MIN_SAMPLES_PER_CLASS = 2500  # target sau augmentation

# ========== PATHS ==========
BASE       = Path('/content')
RAW_AFFECT = BASE / 'affectnet_raw'
RAW_DAISEE = BASE / 'daisee_raw'
RAW_FER    = BASE / 'fer2013_raw'
AFFECT_PREP = BASE / 'affectnet_4class'
FER_PREP    = BASE / 'fer2013_4class'
DAISEE_PREP = BASE / 'daisee_4class'
RUN_DIR     = BASE / 'ultimate_run'
CKPT_DIR    = RUN_DIR / 'checkpoints'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ========== LABELS ==========
LABELS_VI = ['Bu\u1ed3n ch\u00e1n', 'T\u1eadp trung', 'H\u1ee9ng th\u00fa', 'B\u00ecnh th\u01b0\u1eddng']
LABELS_EN = ['Boring', 'Focus', 'Interested', 'Normal']

print('\u2705 Configuration:')
print(f'   AffectNet : {USE_AFFECTNET} (max {AFFECTNET_MAX_PER_CLASS}/class)')
print(f'   FER2013   : {USE_FER_PRETRAIN}')
print(f'   Image size: {IMG_SIZE}x{IMG_SIZE}')
print(f'   Batch size: {BATCH_SIZE}')
for c, n in FRAMES_PER_CLIP.items():
    print(f'   Class {c} ({LABELS_VI[c]}): {n} frames/clip')

In [ ]:
# ============================================================
# CELL 3 — Upload Kaggle Key & Download ALL Datasets
# ============================================================
from google.colab import files

print('Upload kaggle.json t\u1eeb Kaggle Account Settings:')
uploaded = files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# --- DAiSEE (b\u1eaft bu\u1ed9c) ---
print('\n\u2b07\ufe0f Downloading DAiSEE...')
!kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip

# --- AffectNet ---
if USE_AFFECTNET:
    print(f'\n\u2b07\ufe0f Downloading AffectNet ({AFFECTNET_KAGGLE})...')
    !kaggle datasets download -d {AFFECTNET_KAGGLE} -p /content/affectnet_raw --unzip
    print('AffectNet folder structure:')
    !find /content/affectnet_raw -maxdepth 3 -type d | head -25

# --- FER2013 ---
if USE_FER_PRETRAIN:
    print('\n\u2b07\ufe0f Downloading FER2013...')
    !kaggle datasets download -d msambare/fer2013 -p /content/fer2013_raw --unzip

print('\n\u2705 All datasets downloaded.')
!df -h /content | tail -1

In [ ]:
# ============================================================
# CELL 4 — Imports & Core Utilities
# ============================================================
import glob, json, math, shutil, warnings
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

try:
    import albumentations as A
    HAS_ALBUM = True
except ImportError:
    HAS_ALBUM = False

warnings.filterwarnings('ignore')
AUTOTUNE = tf.data.AUTOTUNE
tf.random.set_seed(SEED)

# GPU setup
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print('\u2705 Mixed precision ON')
    except Exception as e:
        print(f'Mixed precision failed: {e}')
print(f'\u2705 GPU: {[g.name for g in gpus] if gpus else "CPU only"}')
print(f'\u2705 Albumentations: {HAS_ALBUM}')

# ========== MAPPINGS ==========

# AffectNet: 0=Neutral 1=Happy 2=Sad 3=Surprise 4=Fear 5=Disgust 6=Anger 7=Contempt
AFFECTNET_CLASS_NAMES = {
    0: 'neutral', 1: 'happy', 2: 'sad', 3: 'surprise',
    4: 'fear', 5: 'disgust', 6: 'anger', 7: 'contempt',
}
# Mapping AffectNet -> Our 4 classes
# Neutral \u0111\u01b0\u1ee3c chia: 60% -> T\u1eadp trung (1), 40% -> B\u00ecnh th\u01b0\u1eddng (3)
AFFECTNET_TO_EMOTION = {
    # AffectNet class -> our class
    1: 2,  # Happy     -> H\u1ee9ng th\u00fa
    2: 0,  # Sad       -> Bu\u1ed3n ch\u00e1n
    3: 2,  # Surprise  -> H\u1ee9ng th\u00fa
    4: 0,  # Fear      -> Bu\u1ed3n ch\u00e1n
    5: 0,  # Disgust   -> Bu\u1ed3n ch\u00e1n
    6: 0,  # Anger     -> Bu\u1ed3n ch\u00e1n
    7: 0,  # Contempt  -> Bu\u1ed3n ch\u00e1n
}
# Neutral (class 0) \u0111\u01b0\u1ee3c x\u1eed l\u00fd ri\u00eang: split gi\u1eefa class 1 v\u00e0 3
NEUTRAL_SPLIT_RATIO = 0.6  # 60% -> T\u1eadp trung, 40% -> B\u00ecnh th\u01b0\u1eddng

# FER2013 mapping (gi\u1eef nguy\u00ean nh\u01b0 c\u0169)
FER2013_TO_EMOTION = {
    'angry': 0, 'disgust': 0, 'fear': 0, 'sad': 0,
    'happy': 2, 'surprise': 2,
    'neutral': 3,
}

# Face detector
FACE_CASCADE = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)


# ========== HELPER FUNCTIONS ==========

def ensure_dirs(root, splits, n_classes=4):
    for s in splits:
        for c in range(n_classes):
            (root / s / str(c)).mkdir(parents=True, exist_ok=True)

def reset_dir(root, splits):
    if root.exists(): shutil.rmtree(root)
    ensure_dirs(root, splits)

def count_images(root, exts=('.jpg', '.jpeg', '.png')):
    result = {}
    for split in ['train', 'val', 'test']:
        d = root / split
        if not d.exists(): continue
        result[split] = {}
        for c in range(NUM_CLASSES):
            cd = d / str(c)
            result[split][c] = sum(1 for f in cd.rglob('*') if f.suffix.lower() in exts) if cd.exists() else 0
    return result

def map_daisee_to_label(row):
    b = row.get('Boredom', row.get('boredom', 0))
    e = row.get('Engagement', row.get('engagement', 0))
    c = row.get('Confusion', row.get('confusion', 0))
    f = row.get('Frustration', row.get('frustration', 0))
    if b >= 2 and e <= 1: return 0
    if e >= 2 and c >= 1 and f < 2: return 2
    if e >= 2: return 1
    return 3

def find_csv(csvs, kw):
    m = [p for p in csvs if kw.lower() in p.lower()]
    return m[0] if m else None

def find_video(clip):
    for ext in ['', '.avi', '.mp4']:
        f = glob.glob(f'/content/daisee_raw/**/{clip}{ext}', recursive=True)
        if f: return f[0]
    return None

def crop_face(frame, pad=24):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = FACE_CASCADE.detectMultiScale(gray, 1.1, 5, minSize=(48, 48))
    if len(faces) > 0:
        areas = [w*h for (_,_,w,h) in faces]
        x,y,w,h = faces[int(np.argmax(areas))]
        return frame[max(0,y-pad):min(frame.shape[0],y+h+pad),
                     max(0,x-pad):min(frame.shape[1],x+w+pad)]
    h,w = frame.shape[:2]
    s = min(h,w)
    return frame[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2]

print('\u2705 Utilities ready.')

In [ ]:
# ============================================================
# CELL 5 — Prepare AffectNet (4-class mapping)
# ============================================================

def find_affectnet_structure(raw_dir):
    """
    Auto-detect AffectNet folder structure.
    Returns dict: {split_name: {affectnet_class_idx: Path}}
    """
    result = {}

    # Pattern 1: train/0..7, val/0..7 (or test/0..7)
    for split in ['train', 'training', 'val', 'validation', 'test']:
        split_dir = None
        for child in raw_dir.rglob('*'):
            if child.is_dir() and child.name.lower() == split:
                # Check if it has numbered subdirs
                numbered = [c for c in child.iterdir() if c.is_dir() and c.name.isdigit()]
                if numbered:
                    split_dir = child
                    break
                # Check for named subdirs (happy, sad, etc.)
                named = [c for c in child.iterdir()
                         if c.is_dir() and c.name.lower() in AFFECTNET_CLASS_NAMES.values()]
                if named:
                    split_dir = child
                    break
        if split_dir:
            out_split = 'train' if split in ('train', 'training') else 'val'
            class_map = {}
            for sub in split_dir.iterdir():
                if not sub.is_dir(): continue
                if sub.name.isdigit():
                    class_map[int(sub.name)] = sub
                else:
                    # Named folder -> find index
                    for idx, name in AFFECTNET_CLASS_NAMES.items():
                        if sub.name.lower() == name:
                            class_map[idx] = sub
                            break
            if class_map:
                result[out_split] = class_map

    # Pattern 2: numbered dirs at root level
    if not result:
        numbered_root = [c for c in raw_dir.iterdir()
                         if c.is_dir() and c.name.isdigit() and int(c.name) <= 7]
        if len(numbered_root) >= 4:
            class_map = {int(c.name): c for c in numbered_root}
            result['train'] = class_map

    return result


def prepare_affectnet(max_per_class=15000):
    """Map AffectNet to our 4-class structure with balanced sampling."""
    reset_dir(AFFECT_PREP, ['train', 'val'])

    structure = find_affectnet_structure(RAW_AFFECT)
    if not structure:
        print('\u26a0\ufe0f Could not detect AffectNet structure!')
        print('Folders found:')
        for p in sorted(RAW_AFFECT.rglob('*')):
            if p.is_dir() and len(list(p.parts)) <= len(RAW_AFFECT.parts) + 3:
                print(f'  {p}')
        return

    print(f'Detected AffectNet structure:')
    for split, classes in structure.items():
        print(f'  {split}: classes {sorted(classes.keys())}')

    stats = Counter()

    for split, class_dirs in structure.items():
        out_split = split  # train -> train, val -> val
        # Max per class is lower for val
        max_n = max_per_class if out_split == 'train' else max_per_class // 5

        for affect_cls, src_dir in sorted(class_dirs.items()):
            images = [f for f in src_dir.rglob('*')
                      if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
            random.shuffle(images)

            if affect_cls == 0:  # Neutral -> split between class 1 and 3
                n_take = min(len(images), max_n)
                split_point = int(n_take * NEUTRAL_SPLIT_RATIO)
                # 60% -> class 1 (T\u1eadp trung)
                for img in tqdm(images[:split_point],
                               desc=f'{out_split} Neutral\u21921', leave=False):
                    dst = AFFECT_PREP / out_split / '1' / f'an_{img.stem}.jpg'
                    shutil.copy2(img, dst)
                    stats[f'{out_split}_1'] += 1
                # 40% -> class 3 (B\u00ecnh th\u01b0\u1eddng)
                for img in tqdm(images[split_point:n_take],
                               desc=f'{out_split} Neutral\u21923', leave=False):
                    dst = AFFECT_PREP / out_split / '3' / f'an_{img.stem}.jpg'
                    shutil.copy2(img, dst)
                    stats[f'{out_split}_3'] += 1
            else:
                our_cls = AFFECTNET_TO_EMOTION.get(affect_cls)
                if our_cls is None:
                    continue
                n_take = min(len(images), max_n)
                label_name = AFFECTNET_CLASS_NAMES[affect_cls]
                for img in tqdm(images[:n_take],
                               desc=f'{out_split} {label_name}\u2192{our_cls}', leave=False):
                    dst = AFFECT_PREP / out_split / str(our_cls) / f'an_{label_name}_{img.stem}.jpg'
                    if dst.exists():
                        dst = AFFECT_PREP / out_split / str(our_cls) / f'an_{label_name}_{abs(hash(str(img)))%100000}.jpg'
                    shutil.copy2(img, dst)
                    stats[f'{out_split}_{our_cls}'] += 1

    affect_counts = count_images(AFFECT_PREP)
    print('\n\u2705 AffectNet prepared:')
    for split, counts in affect_counts.items():
        total = sum(counts.values())
        detail = '  '.join(f'{LABELS_VI[c]}={n}' for c, n in sorted(counts.items()))
        print(f'   {split}: {detail}  (total: {total:,})')


if USE_AFFECTNET:
    prepare_affectnet(AFFECTNET_MAX_PER_CLASS)
else:
    print('Skipping AffectNet.')

In [ ]:
# ============================================================
# CELL 6 — Prepare FER2013
# ============================================================

def prepare_fer2013():
    reset_dir(FER_PREP, ['train', 'val'])
    aliases = {'train': {'train', 'training'}, 'val': {'test', 'validation', 'val'}}
    for split_name, alts in aliases.items():
        src = None
        for child in RAW_FER.rglob('*'):
            if child.is_dir() and child.name.lower() in alts:
                src = child; break
        if src is None:
            raise FileNotFoundError(f'FER2013 {split_name} not found')
        for emo_dir in src.iterdir():
            if not emo_dir.is_dir(): continue
            mapped = FER2013_TO_EMOTION.get(emo_dir.name.lower())
            if mapped is None: continue
            dst_dir = FER_PREP / split_name / str(mapped)
            for img in emo_dir.rglob('*'):
                if img.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
                target = dst_dir / img.name
                if target.exists():
                    target = dst_dir / f'{img.stem}_{abs(hash(str(img)))%100000}{img.suffix}'
                shutil.copy2(img, target)

if USE_FER_PRETRAIN:
    prepare_fer2013()
    fer_counts = count_images(FER_PREP)
    print('\u2705 FER2013 prepared:')
    for split, counts in fer_counts.items():
        total = sum(counts.values())
        detail = '  '.join(f'{LABELS_VI[c]}={n}' for c, n in sorted(counts.items()))
        print(f'   {split}: {detail}  (total: {total:,})')
else:
    print('Skipping FER2013.')

In [ ]:
# ============================================================
# CELL 7 — Prepare DAiSEE with Adaptive Frame Extraction
# ============================================================

def extract_frames(video_path, label, split, clip_id):
    n_frames = FRAMES_PER_CLIP.get(label, 10)
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return 0
    n = min(n_frames, total)
    pts = np.linspace(0, total - 1, n, dtype=int)
    saved = 0
    for i, pt in enumerate(pts):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(pt))
        ok, frame = cap.read()
        if not ok or frame is None: continue
        face = crop_face(frame)
        if face.size == 0: continue
        face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        out = DAISEE_PREP / split / str(label) / f'{clip_id}_{i:03d}.jpg'
        cv2.imwrite(str(out), face, [cv2.IMWRITE_JPEG_QUALITY, 94])
        saved += 1
    cap.release()
    return saved


def prepare_daisee():
    reset_dir(DAISEE_PREP, ['train', 'val', 'test'])
    csvs = glob.glob('/content/daisee_raw/**/*.csv', recursive=True)
    df_train = pd.read_csv(find_csv(csvs, 'train'))
    df_val   = pd.read_csv(find_csv(csvs, 'validat'))
    df_test  = pd.read_csv(find_csv(csvs, 'test'))

    for df, split in [(df_train, 'train'), (df_val, 'val'), (df_test, 'test')]:
        df['emotion_label'] = df.apply(map_daisee_to_label, axis=1)
        dist = df['emotion_label'].value_counts().to_dict()
        print(f'{split} clips: {dist}')
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f'Extract {split}'):
            clip = row.get('ClipID', row.get('Clip', row.get('clip')))
            if clip is None: continue
            vpath = find_video(str(clip))
            if vpath is None: continue
            extract_frames(vpath, int(row['emotion_label']), split, idx)

prepare_daisee()
daisee_counts = count_images(DAISEE_PREP)
print('\n\u2705 DAiSEE prepared:')
for split, counts in daisee_counts.items():
    total = sum(counts.values())
    detail = '  '.join(f'{LABELS_VI[c]}={n}' for c, n in sorted(counts.items()))
    print(f'   {split}: {detail}  (total: {total:,})')

In [ ]:
# ============================================================
# CELL 8 — Offline Augmentation for DAiSEE Minority Classes
# ============================================================

def augment_cv2(img):
    t = random.randint(0, 5)
    h, w = img.shape[:2]
    if t == 0:
        img = cv2.flip(img, 1)
    elif t == 1:
        M = cv2.getRotationMatrix2D((w/2, h/2), random.uniform(-20, 20), 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    elif t == 2:
        img = np.clip(img.astype(np.float32) * random.uniform(0.7, 1.3), 0, 255).astype(np.uint8)
    elif t == 3:
        m = np.mean(img)
        img = np.clip((img.astype(np.float32) - m) * random.uniform(0.7, 1.4) + m, 0, 255).astype(np.uint8)
    elif t == 4:
        img = cv2.GaussianBlur(img, (random.choice([3,5]),)*2, 0)
    elif t == 5:
        f = random.uniform(0.8, 0.95)
        nh, nw = int(h*f), int(w*f)
        y, x = random.randint(0, h-nh), random.randint(0, w-nw)
        img = cv2.resize(img[y:y+nh, x:x+nw], (w, h))
    return img

def augment_album(img):
    t = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=20,
                           border_mode=cv2.BORDER_REFLECT_101, p=0.7),
        A.OneOf([
            A.RandomBrightnessContrast(0.3, 0.3, p=1),
            A.HueSaturationValue(10, 25, 25, p=1),
        ], p=0.6),
        A.OneOf([
            A.GaussianBlur((3, 5), p=1),
            A.GaussNoise(var_limit=(5, 30), p=1),
        ], p=0.3),
        A.CoarseDropout(max_holes=4, max_height=20, max_width=20, fill_value=0, p=0.2),
    ])
    return t(image=img)['image']


def augment_minority(root, split='train', min_n=2500):
    aug_fn = augment_album if HAS_ALBUM else augment_cv2
    print(f'Augmentation engine: {"Albumentations" if HAS_ALBUM else "OpenCV"}')

    for c in range(NUM_CLASSES):
        cdir = root / split / str(c)
        existing = list(cdir.glob('*.jpg'))
        n = len(existing)
        if n >= min_n:
            print(f'   {LABELS_VI[c]}: {n} \u2265 {min_n} \u2192 SKIP')
            continue
        if n == 0:
            print(f'   {LABELS_VI[c]}: 0 images! SKIP')
            continue
        need = min_n - n
        print(f'   {LABELS_VI[c]}: {n} \u2192 generating {need} augmented images...')
        for i in tqdm(range(need), desc=f'Aug {LABELS_VI[c]}', leave=False):
            src = random.choice(existing)
            img = cv2.imread(str(src))
            if img is None: continue
            aug = aug_fn(img)
            cv2.imwrite(str(cdir / f'aug_{c}_{i:05d}.jpg'), aug, [cv2.IMWRITE_JPEG_QUALITY, 92])


augment_minority(DAISEE_PREP, 'train', MIN_SAMPLES_PER_CLASS)

final_counts = count_images(DAISEE_PREP)
print('\n\u2705 DAiSEE after augmentation:')
for split, counts in final_counts.items():
    total = sum(counts.values())
    detail = '  '.join(f'{LABELS_VI[c]}={n}' for c, n in sorted(counts.items()))
    print(f'   {split}: {detail}  (total: {total:,})')

In [ ]:
# ============================================================
# CELL 9 — Visualize Dataset Distributions
# ============================================================

datasets_to_plot = []
if USE_AFFECTNET:
    datasets_to_plot.append(('AffectNet', count_images(AFFECT_PREP)))
if USE_FER_PRETRAIN:
    datasets_to_plot.append(('FER2013', count_images(FER_PREP)))
datasets_to_plot.append(('DAiSEE (augmented)', count_images(DAISEE_PREP)))

colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71']

fig, axes = plt.subplots(1, len(datasets_to_plot), figsize=(7 * len(datasets_to_plot), 5))
if len(datasets_to_plot) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, datasets_to_plot):
    train_counts = info.get('train', {})
    vals = [train_counts.get(c, 0) for c in range(NUM_CLASSES)]
    total = sum(vals)
    bars = ax.bar(LABELS_VI, vals, color=colors)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f'{v:,}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'{name} Train ({total:,})', fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.15 if vals else 1)

fig.suptitle('Dataset Distributions', fontsize=15, fontweight='bold')
fig.tight_layout()
fig.savefig(RUN_DIR / 'data_distribution.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 10 — Model Architecture & Training Functions
# ============================================================

def make_ds(root, split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        root / split, label_mode='categorical',
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED,
    ).prefetch(AUTOTUNE)


def smart_class_weights(split_dir):
    labels = []
    for c in range(NUM_CLASSES):
        n = sum(1 for f in (split_dir / str(c)).rglob('*.jpg'))
        labels.extend([c] * n)
    labels = np.array(labels, dtype=np.int32)
    classes = np.unique(labels)
    w = compute_class_weight('balanced', classes=classes, y=labels)
    cw = {int(c): float(v) for c, v in zip(classes, w)}
    counts = Counter(labels)
    mx = max(counts.values())
    for c in range(NUM_CLASSES):
        if counts.get(c, 0) > 0:
            ratio = mx / counts[c]
            if ratio > 5: cw[c] *= 1.5
            elif ratio > 2: cw[c] *= 1.2
        cw[c] = max(cw.get(c, 1.0), 0.4)
    return cw


def build_loss():
    focal = getattr(tf.keras.losses, 'CategoricalFocalCrossentropy', None)
    if focal:
        return focal(alpha=0.25, gamma=2.5, label_smoothing=0.05)
    return tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)


def build_model():
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
    # Online augmentation
    x = tf.keras.layers.RandomFlip('horizontal')(inp)
    x = tf.keras.layers.RandomRotation(0.10)(x)
    x = tf.keras.layers.RandomZoom(0.15)(x)
    x = tf.keras.layers.RandomTranslation(0.10, 0.10)(x)
    x = tf.keras.layers.RandomContrast(0.2)(x)
    # Backbone
    x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
    backbone = tf.keras.applications.EfficientNetV2B0(
        include_top=False, weights='imagenet', input_tensor=x, pooling='avg',
    )
    backbone.trainable = False
    # Head
    x = tf.keras.layers.BatchNormalization()(backbone.output)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(512, activation='swish',
            kernel_regularizer=tf.keras.regularizers.l2(2e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(256, activation='swish',
            kernel_regularizer=tf.keras.regularizers.l2(2e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax',
            dtype='float32', name='emotion')(x)
    m = tf.keras.Model(inp, out, name='emotion_v2')
    m.backbone = backbone
    return m


def unfreeze(model, fraction):
    bb = model.backbone
    bb.trainable = True
    total = len(bb.layers)
    cut = int(total * (1.0 - fraction))
    for l in bb.layers[:cut]: l.trainable = False
    for l in bb.layers[cut:]:
        if not isinstance(l, tf.keras.layers.BatchNormalization):
            l.trainable = True
    tr = sum(1 for l in bb.layers if l.trainable)
    print(f'   Backbone: {tr}/{total} layers trainable ({fraction*100:.0f}%)')


def freeze_backbone(model):
    model.backbone.trainable = False
    print('   Backbone frozen.')


def compile_m(model, lr, steps=None):
    if steps and steps > 0:
        sched = tf.keras.optimizers.schedules.CosineDecay(lr, steps, alpha=1e-6)
        opt = tf.keras.optimizers.Adam(sched)
    else:
        opt = tf.keras.optimizers.Adam(lr)
    model.compile(optimizer=opt, loss=build_loss(), metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc', multi_label=False),
    ])


def cbs(name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(CKPT_DIR / f'{name}.keras'),
            monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=6,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.4, patience=3,
            min_lr=1e-7, verbose=1),
        tf.keras.callbacks.CSVLogger(str(RUN_DIR / f'{name}.csv')),
    ]


print('\u2705 Model functions ready.')
model = build_model()
total_params = model.count_params()
trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable:,}')

In [ ]:
# ============================================================
# CELL 11 — TRAINING PIPELINE
# ============================================================

histories = {}

# ===========================================================
# STAGE A: AffectNet Pretrain
# ===========================================================
if USE_AFFECTNET:
    print('\n' + '='*65)
    print('  STAGE A1: AffectNet — Head Training (backbone frozen)')
    print('='*65)
    an_train = make_ds(AFFECT_PREP, 'train', shuffle=True)
    an_val   = make_ds(AFFECT_PREP, 'val')
    an_cw    = smart_class_weights(AFFECT_PREP / 'train')
    print(f'Class weights: {an_cw}')
    print(f'Steps/epoch: {len(an_train)}')

    freeze_backbone(model)
    compile_m(model, lr=8e-4, steps=len(an_train) * AFFECTNET_EPOCHS_HEAD)
    h = model.fit(an_train, validation_data=an_val,
                  epochs=AFFECTNET_EPOCHS_HEAD,
                  class_weight=an_cw, callbacks=cbs('affect_head'), verbose=1)
    histories['affect_head'] = h

    print('\n' + '='*65)
    print('  STAGE A2: AffectNet — Fine-tune (25% backbone)')
    print('='*65)
    unfreeze(model, 0.25)
    compile_m(model, lr=1e-4, steps=len(an_train) * AFFECTNET_EPOCHS_FINETUNE)
    h = model.fit(an_train, validation_data=an_val,
                  epochs=AFFECTNET_EPOCHS_FINETUNE,
                  class_weight=an_cw, callbacks=cbs('affect_ft'), verbose=1)
    histories['affect_ft'] = h
    print('\u2705 AffectNet pretrain done.\n')

    # Gi\u1ea3i ph\u00f3ng RAM
    del an_train, an_val
    import gc; gc.collect()


# ===========================================================
# STAGE B: FER2013 Pretrain
# ===========================================================
if USE_FER_PRETRAIN:
    print('\n' + '='*65)
    print('  STAGE B1: FER2013 — Head Training')
    print('='*65)
    fer_train = make_ds(FER_PREP, 'train', shuffle=True)
    fer_val   = make_ds(FER_PREP, 'val')
    fer_cw    = smart_class_weights(FER_PREP / 'train')
    print(f'Class weights: {fer_cw}')

    freeze_backbone(model)
    compile_m(model, lr=6e-4, steps=len(fer_train) * FER_EPOCHS_HEAD)
    h = model.fit(fer_train, validation_data=fer_val,
                  epochs=FER_EPOCHS_HEAD,
                  class_weight=fer_cw, callbacks=cbs('fer_head'), verbose=1)
    histories['fer_head'] = h

    print('\n' + '='*65)
    print('  STAGE B2: FER2013 — Fine-tune (25% backbone)')
    print('='*65)
    unfreeze(model, 0.25)
    compile_m(model, lr=8e-5, steps=len(fer_train) * FER_EPOCHS_FINETUNE)
    h = model.fit(fer_train, validation_data=fer_val,
                  epochs=FER_EPOCHS_FINETUNE,
                  class_weight=fer_cw, callbacks=cbs('fer_ft'), verbose=1)
    histories['fer_ft'] = h
    print('\u2705 FER2013 pretrain done.\n')

    del fer_train, fer_val
    import gc; gc.collect()


# ===========================================================
# STAGE C: DAiSEE Training (3 stages)
# ===========================================================
print('\n' + '='*65)
print('  STAGE C1: DAiSEE — Head Training (backbone frozen)')
print('='*65)
ds_train = make_ds(DAISEE_PREP, 'train', shuffle=True)
ds_val   = make_ds(DAISEE_PREP, 'val')
ds_test  = make_ds(DAISEE_PREP, 'test')
ds_cw    = smart_class_weights(DAISEE_PREP / 'train')
print(f'Class weights: {ds_cw}')
print(f'Steps/epoch: {len(ds_train)}')

freeze_backbone(model)
compile_m(model, lr=5e-4, steps=len(ds_train) * DAISEE_EPOCHS_HEAD)
h = model.fit(ds_train, validation_data=ds_val,
              epochs=DAISEE_EPOCHS_HEAD,
              class_weight=ds_cw, callbacks=cbs('daisee_head'), verbose=1)
histories['daisee_head'] = h

print('\n' + '='*65)
print('  STAGE C2: DAiSEE — Fine-tune (30% backbone)')
print('='*65)
unfreeze(model, 0.30)
compile_m(model, lr=5e-5, steps=len(ds_train) * DAISEE_EPOCHS_FT1)
h = model.fit(ds_train, validation_data=ds_val,
              epochs=DAISEE_EPOCHS_FT1,
              class_weight=ds_cw, callbacks=cbs('daisee_ft1'), verbose=1)
histories['daisee_ft1'] = h

print('\n' + '='*65)
print('  STAGE C3: DAiSEE — Deep Fine-tune (60% backbone)')
print('='*65)
unfreeze(model, 0.60)
compile_m(model, lr=1e-5, steps=len(ds_train) * DAISEE_EPOCHS_FT2)
h = model.fit(ds_train, validation_data=ds_val,
              epochs=DAISEE_EPOCHS_FT2,
              class_weight=ds_cw, callbacks=cbs('daisee_ft2'), verbose=1)
histories['daisee_ft2'] = h

print('\n\u2705 ALL TRAINING COMPLETE!')

In [ ]:
# ============================================================
# CELL 12 — Comprehensive Evaluation
# ============================================================

# Load best checkpoint
best_ckpt = CKPT_DIR / 'daisee_ft2.keras'
if best_ckpt.exists():
    print(f'Loading best checkpoint: {best_ckpt}')
    best_model = tf.keras.models.load_model(best_ckpt)
else:
    print('Using current model weights.')
    best_model = model

# Collect predictions
y_true_all, y_pred_all = [], []
for imgs, lbls in tqdm(ds_test, desc='Evaluating'):
    probs = best_model.predict(imgs, verbose=0)
    y_true_all.append(np.argmax(lbls.numpy(), axis=1))
    y_pred_all.append(np.argmax(probs, axis=1))

y_true = np.concatenate(y_true_all)
y_pred = np.concatenate(y_pred_all)

# Metrics
report = classification_report(
    y_true, y_pred, labels=list(range(NUM_CLASSES)),
    target_names=LABELS_VI, output_dict=True, zero_division=0)
bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print('\n' + '='*60)
print('  EVALUATION RESULTS')
print('='*60)
print(classification_report(
    y_true, y_pred, labels=list(range(NUM_CLASSES)),
    target_names=LABELS_VI, zero_division=0))
print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'  Macro F1-Score    : {macro_f1:.4f}')
print(f'  Weighted F1-Score : {weighted_f1:.4f}')

# --- Confusion Matrices ---
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS_VI, yticklabels=LABELS_VI, ax=axes[0])
axes[0].set_xlabel('D\u1ef1 \u0111o\u00e1n'); axes[0].set_ylabel('Th\u1ef1c t\u1ebf')
axes[0].set_title('Raw Counts')

cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=LABELS_VI, yticklabels=LABELS_VI, ax=axes[1], vmin=0, vmax=1)
axes[1].set_xlabel('D\u1ef1 \u0111o\u00e1n'); axes[1].set_ylabel('Th\u1ef1c t\u1ebf')
axes[1].set_title('Normalized (Recall)')

fig.suptitle(f'Balanced Acc: {bal_acc:.3f} | Macro F1: {macro_f1:.3f}',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(RUN_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

# --- Per-class metrics bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(NUM_CLASSES)
w = 0.25
prec = [report[LABELS_VI[i]]['precision'] for i in range(NUM_CLASSES)]
rec = [report[LABELS_VI[i]]['recall'] for i in range(NUM_CLASSES)]
f1s = [report[LABELS_VI[i]]['f1-score'] for i in range(NUM_CLASSES)]
ax.bar(x_pos - w, prec, w, label='Precision', color='#3498db')
ax.bar(x_pos, rec, w, label='Recall', color='#e74c3c')
ax.bar(x_pos + w, f1s, w, label='F1', color='#2ecc71')
ax.set_xticks(x_pos); ax.set_xticklabels(LABELS_VI)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Per-Class Metrics', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / 'per_class_metrics.png', dpi=160)
plt.show()

In [ ]:
# ============================================================
# CELL 13 — Training Curves (All Stages)
# ============================================================

all_acc, all_vacc, all_loss, all_vloss = [], [], [], []
boundaries, stage_labels = [], []
offset = 0

for name, h in histories.items():
    hist = h.history
    n = len(hist.get('accuracy', []))
    if n == 0: continue
    all_acc.extend(hist.get('accuracy', []))
    all_vacc.extend(hist.get('val_accuracy', []))
    all_loss.extend(hist.get('loss', []))
    all_vloss.extend(hist.get('val_loss', []))
    boundaries.append(offset)
    stage_labels.append(name)
    offset += n

epochs = range(1, len(all_acc) + 1)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].plot(epochs, all_acc, 'b-', lw=1.5, label='Train')
axes[0].plot(epochs, all_vacc, 'orange', lw=1.5, label='Val')
axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')

axes[1].plot(epochs, all_loss, 'b-', lw=1.5, label='Train')
axes[1].plot(epochs, all_vloss, 'orange', lw=1.5, label='Val')
axes[1].set_title('Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')

stage_colors = ['#e74c3c', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#e67e22', '#95a5a6']
for i, (b, lbl) in enumerate(zip(boundaries, stage_labels)):
    if b > 0:
        c = stage_colors[i % len(stage_colors)]
        for ax in axes:
            ax.axvline(b + 1, color=c, ls='--', alpha=0.7, lw=1.2)
        axes[0].text(b + 1.5, max(all_acc) * 0.95, lbl, fontsize=7, rotation=90, color=c)

for ax in axes:
    ax.grid(alpha=0.25); ax.legend(fontsize=10)

fig.suptitle('Full Training History', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_curves.png', dpi=160)
plt.show()

In [ ]:
# ============================================================
# CELL 14 — Export Model & Save Summary
# ============================================================

final_h5 = RUN_DIR / 'emotion_model_daisee_best.h5'
final_keras = RUN_DIR / 'emotion_model_daisee_best.keras'
best_model.save(final_h5)
best_model.save(final_keras)
print(f'\u2705 Model .h5   : {final_h5}')
print(f'\u2705 Model .keras: {final_keras}')

# Serialize histories
hist_serial = {}
for stage, h in histories.items():
    hist_serial[stage] = {k: [float(v) for v in vals] for k, vals in h.history.items()}

summary = {
    'config': {
        'use_affectnet': USE_AFFECTNET,
        'affectnet_dataset': AFFECTNET_KAGGLE,
        'affectnet_max_per_class': AFFECTNET_MAX_PER_CLASS,
        'use_fer': USE_FER_PRETRAIN,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'min_samples_per_class': MIN_SAMPLES_PER_CLASS,
        'frames_per_clip': FRAMES_PER_CLIP,
    },
    'datasets': {
        'daisee': count_images(DAISEE_PREP),
    },
    'class_weights_daisee': ds_cw,
    'evaluation': {
        'balanced_accuracy': float(bal_acc),
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1),
        'classification_report': report,
        'confusion_matrix': cm.tolist(),
    },
    'histories': hist_serial,
}

with open(RUN_DIR / 'training_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'\u2705 Summary: {RUN_DIR / "training_summary.json"}')

# Final scoreboard
print('\n' + '='*60)
print('  \ud83c\udfaf FINAL SCOREBOARD')
print('='*60)
print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'  Macro F1-Score    : {macro_f1:.4f}')
print(f'  Weighted F1-Score : {weighted_f1:.4f}')
print(f'\n  Per-class:')
for i in range(NUM_CLASSES):
    r = report[LABELS_VI[i]]
    print(f'    {LABELS_VI[i]:12s}: P={r["precision"]:.3f}  R={r["recall"]:.3f}  F1={r["f1-score"]:.3f}  (n={r["support"]})')
print('='*60)

In [ ]:
# ============================================================
# CELL 15 — Download Results
# ============================================================
%cd /content
!zip -r ultimate_run.zip ultimate_run > /dev/null

from google.colab import files
files.download('/content/ultimate_run.zip')

print('\n\u2705 Download started!')
print('\nFiles:')
!find ultimate_run -type f | sort

---

## 📝 Sau khi download

1. Gi\u1ea3i n\u00e9n `ultimate_run.zip`
2. Copy `emotion_model_daisee_best.h5` v\u00e0o `models/` c\u1ee7a project
3. Ch\u1ea1y `python main.py` \u0111\u1ec3 test webcam

### Tips n\u1ebfu k\u1ebft qu\u1ea3 ch\u01b0a nh\u01b0 \u00fd:
- T\u0103ng `AFFECTNET_MAX_PER_CLASS` l\u00ean 20000-25000
- T\u0103ng `MIN_SAMPLES_PER_CLASS` l\u00ean 4000-5000
- T\u0103ng `FRAMES_PER_CLIP[0]` v\u00e0 `[3]` l\u00ean 60-80
- N\u1ebfu c\u00f3 A100: t\u0103ng `IMG_SIZE=300`, `BATCH_SIZE=64`
- T\u0103ng epochs DAiSEE n\u1ebfu c\u1ea7n: `DAISEE_EPOCHS_FT1=25`, `FT2=20`
